# CS 228: export data for running on a Mac (CPU, no GPU quota)

Packs the prebuilt `data_pad2` crops and the two CelebA-Spoof label files into one zip, `mac_data.zip`, so the checkpoint can run locally with `scripts/run_checkpoint_mac.sh`.

**Inputs (Add Input):** your notebook `cs228-data` (Notebooks tab) and the dataset `celeba-spoof-for-face-antispoofing` (Datasets tab). **Accelerator: None.** Internet not needed.
**Save Version -> Save & Run All (Commit).** About 5 minutes. Then download `mac_data.zip` from Output.

The zip holds research-licensed CelebA-Spoof data: keep it on your machine, never commit it.

In [ ]:
import glob, os, shutil
# 1. Prebuilt crops from the cs228-data notebook output (shallow search)
found = []
for depth in range(1, 6):
    found += glob.glob('/kaggle/input/' + '*/' * (depth - 1) + 'data_pad2')
print('data_pad2:', found)
assert found, 'Attach your cs228-data notebook: Add Input -> Notebooks -> cs228-data'
DATA = found[0]

In [ ]:
# 2. Label files: <root>/metas/intra_test/{train,test}_label.json (depth-limited, never a recursive walk)
meta = []
for depth in range(0, 7):
    meta = glob.glob(os.path.join('/kaggle/input', *(['*'] * depth), 'metas', 'intra_test'))
    if meta: break
print('labels:', meta)
assert meta, 'Attach the dataset: Add Input -> Datasets -> celeba-spoof-for-face-antispoofing'
META = meta[0]

In [ ]:
# 3. Pack into one zip
os.chdir('/kaggle/working')
shutil.rmtree('export', ignore_errors=True)
os.makedirs('export/celeba_meta/metas/intra_test')
for f in ('train_label.json', 'test_label.json'):
    shutil.copy(os.path.join(META, f), 'export/celeba_meta/metas/intra_test/')
shutil.copytree(DATA, 'export/data_pad2')
for s in ('train', 'val', 'test'):
    for c in ('live', 'spoof'):
        print(s, c, len(os.listdir(f'export/data_pad2/{s}/{c}')))
!cd export && zip -qr ../mac_data.zip data_pad2 celeba_meta
shutil.rmtree('export')
!ls -lh mac_data.zip